# Scorer Quality

Leaderboards for the scorer evaluation metrics PyRIT already tracks under
`pyrit/datasets/scorer_evals/`. See [Scorer Metrics](../code/scoring/4_scorer_metrics.ipynb) for
what each metric means and how these numbers are produced.

## Objective Scorer Leaderboard

Objective scorers answer a true/false question (e.g. "was the objective achieved?"). Ranked by
F1 score, the harmonic mean of precision and recall.

In [ ]:
import html

import pandas as pd
from IPython.display import HTML

from pyrit.score import get_all_objective_metrics
from pyrit.setup import IN_MEMORY, initialize_pyrit_async

await initialize_pyrit_async(memory_db_type=IN_MEMORY, silent=True)  # type: ignore

# Static (non-interactive) dark leaderboard-card styling shared by every table on this page.
# MyST renders notebook HTML output via React's `dangerouslySetInnerHTML`, and browsers never
# execute <script> tags inserted that way, so cards are pre-sorted/pre-formatted here rather than
# offering the live search/sort a browser-side script would normally provide.
_CARD_STYLE = """<style>
.pyrit-leaderboard-card {
  font-family: "SFMono-Regular", Consolas, "Liberation Mono", monospace;
  background: #0d1117;
  color: #e6edf3;
  border: 1px solid #30363d;
  border-radius: 6px;
  max-width: 900px;
  overflow: hidden;
}
.pyrit-leaderboard-card .lb-head {
  display: flex;
  align-items: center;
  justify-content: space-between;
  gap: 10px;
  padding: 10px 16px;
  background: #161b22;
  border-bottom: 1px solid #30363d;
}
.pyrit-leaderboard-card .lb-head h4 {
  margin: 0;
  font-size: 11px;
  text-transform: uppercase;
  letter-spacing: 0.08em;
  color: #8b949e;
  font-weight: 600;
}
.pyrit-leaderboard-card .lb-meta { font-size: 11px; color: #8b949e; white-space: nowrap; }
.pyrit-leaderboard-card table { width: 100%; border-collapse: collapse; font-size: 12.5px; }
.pyrit-leaderboard-card th {
  text-align: left;
  padding: 8px 14px;
  font-size: 10px;
  text-transform: uppercase;
  letter-spacing: 0.06em;
  color: #3fb950;
  border-bottom: 1px solid #30363d;
  white-space: nowrap;
}
.pyrit-leaderboard-card th.lb-rank { text-align: center; }
.pyrit-leaderboard-card td { padding: 8px 14px; border-bottom: 1px solid #21262d; }
.pyrit-leaderboard-card td.lb-rank { color: #6e7681; text-align: center; }
.pyrit-leaderboard-card .lb-note {
  font-size: 11px;
  color: #8b949e;
  padding: 8px 16px;
  border-top: 1px solid #30363d;
}
</style>
"""


def _format_cell(value: object, *, as_percent: bool) -> str:
    """Format a single leaderboard cell value as display text, escaping any string content."""
    if isinstance(value, (int, float)) and not isinstance(value, bool):
        if as_percent:
            return f"{float(value):.0%}"
        return f"{value:.2f}" if isinstance(value, float) else str(value)
    return html.escape(str(value))


def render_leaderboard_card(
    df: pd.DataFrame, *, title: str, note: str, percent_columns: frozenset[str] = frozenset()
) -> HTML:
    """Render a pre-sorted DataFrame as a static dark leaderboard card (HTML/CSS only, no JS)."""
    header_cells = "".join(f"<th>{html.escape(str(column))}</th>" for column in df.columns)
    body_rows = "".join(
        "<tr><td class='lb-rank'>{rank}</td>{cells}</tr>".format(
            rank=rank,
            cells="".join(
                f"<td>{_format_cell(value, as_percent=column in percent_columns)}</td>" for column, value in row.items()
            ),
        )
        for rank, (_, row) in enumerate(df.iterrows(), start=1)
    )
    row_count = len(df)
    meta = f"{row_count} row{'s' if row_count != 1 else ''}"
    card = (
        f'{_CARD_STYLE}<div class="pyrit-leaderboard-card">'
        f'<div class="lb-head"><h4>{html.escape(title)}</h4><span class="lb-meta">{meta}</span></div>'
        f'<table><thead><tr><th class="lb-rank">#</th>{header_cells}</tr></thead>'
        f"<tbody>{body_rows}</tbody></table>"
        f'<div class="lb-note">{html.escape(note)}</div></div>'
    )
    return HTML(card)


objective_metrics = get_all_objective_metrics()
objective_metrics.sort(key=lambda entry: entry.metrics.f1_score, reverse=True)

objective_rows = [
    {
        "Name": entry.scorer_identifier.unique_name,
        "Accuracy": entry.metrics.accuracy,
        "F1 Score": entry.metrics.f1_score,
        "Precision": entry.metrics.precision,
        "Recall": entry.metrics.recall,
        "Samples": entry.metrics.num_responses,
    }
    for entry in objective_metrics
]

objective_df = pd.DataFrame(objective_rows)
render_leaderboard_card(
    objective_df,
    title="Objective Scorer Leaderboard",
    note="All metrics computed against human-labeled ground truth. Ranked by F1 (higher is better across all "
    "four metrics).",
    percent_columns=frozenset({"Accuracy", "F1 Score", "Precision", "Recall"}),
)

Auto-discovered plaintext environment file ./.pyrit/.env will be loaded. Azure Key Vault through env_akv_ref is more secure for shared or deployed secrets; use .env.local only for deliberate local overrides. To inspect a resolved AKV-only configuration from a source checkout, run `python -m build_scripts.export_akv_environment`; it writes ~/.pyrit/.env_akv.


#,Name,Accuracy,F1 Score,Precision,Recall,Samples
1,TrueFalseInverterScorer::e7af90c2,90%,89%,89%,90%,395
2,TrueFalseInverterScorer::9e875a98,89%,89%,88%,91%,395
3,TrueFalseInverterScorer::9355fe9c,88%,88%,87%,88%,395
4,TrueFalseInverterScorer::f31c9af8,88%,87%,91%,84%,395
5,TrueFalseInverterScorer::7383238a,85%,86%,79%,94%,376
6,TrueFalseInverterScorer::4c10ed71,85%,85%,79%,93%,395
7,TrueFalseInverterScorer::cc1b3ff2,79%,83%,71%,99%,376
8,TrueFalseInverterScorer::4c6b1acf,78%,82%,70%,99%,376
9,SelfAskTrueFalseScorer::b0079ec4,80%,77%,87%,69%,395
10,SelfAskTrueFalseScorer::f4f59053,79%,76%,88%,66%,395


## Harm Scorer Leaderboard

Harm scorers produce a severity score (0.0-1.0). Ranked by `krippendorff_alpha_combined` —
agreement between the model's scores and human raters, ranging from -1.0 (systematic
disagreement) to 1.0 (perfect agreement) — across every harm category PyRIT currently has
metrics for. Alpha isn't comparable *across* categories (each has its own human-labeled
dataset), so treat this as one leaderboard per category, stacked into a single table for
convenience.

In [ ]:
from pyrit.common.path import SCORER_EVALS_HARM_PATH
from pyrit.score import get_all_harm_metrics

# Harm categories are discovered from the files present on disk rather than a hardcoded list,
# so a newly added category shows up here without a code change.
harm_categories = sorted(
    path.name.removesuffix("_metrics.jsonl") for path in SCORER_EVALS_HARM_PATH.glob("*_metrics.jsonl")
)

harm_metrics = [
    (harm_category, entry)
    for harm_category in harm_categories
    for entry in get_all_harm_metrics(harm_category=harm_category)
]
harm_metrics.sort(key=lambda item: item[1].metrics.krippendorff_alpha_combined, reverse=True)

harm_rows = [
    {
        "Name": entry.scorer_identifier.unique_name,
        "Harm Category": harm_category,
        "MAE": entry.metrics.mean_absolute_error,
        "Alpha Combined": entry.metrics.krippendorff_alpha_combined,
        "Alpha Humans": entry.metrics.krippendorff_alpha_humans,
        "Alpha Model": entry.metrics.krippendorff_alpha_model,
        "Samples": entry.metrics.num_responses,
    }
    for harm_category, entry in harm_metrics
]

harm_df = pd.DataFrame(harm_rows)
render_leaderboard_card(
    harm_df,
    title="Harm Scorer Leaderboard",
    note="MAE: lower is better (0-1 scale). Alpha = Krippendorff's alpha: higher is better (agreement between "
    "scorer and ground truth). Not comparable across harm categories.",
)

#,Name,Harm Category,MAE,Alpha Combined,Alpha Humans,Alpha Model,Samples
1,SelfAskLikertScorer::ce5da81b,sexual,0.13,0.90,None,0.98,78
2,AzureContentFilterScorer::1a9b9789,hate_speech,0.17,0.86,None,1.00,59
3,SelfAskLikertScorer::ce31ba14,hate_speech,0.17,0.85,None,0.95,59
4,SelfAskLikertScorer::772e51c8,violence,0.16,0.85,None,0.93,96
5,SelfAskLikertScorer::1607ea13,hate_speech,0.17,0.85,None,0.95,59
6,SelfAskLikertScorer::8e9637f6,hate_speech,0.18,0.85,None,0.95,59
7,SelfAskLikertScorer::118336dd,violence,0.17,0.85,None,0.96,96
8,SelfAskLikertScorer::67df3fb9,violence,0.18,0.84,None,0.97,96
9,AzureContentFilterScorer::9e2a1052,sexual,0.19,0.84,None,1.00,78
10,SelfAskLikertScorer::0317dc04,sexual,0.15,0.82,None,0.87,78


## Note on scope

`get_all_objective_metrics()` reads `objective/objective_achieved_metrics.jsonl` only, matching
how it's documented and used elsewhere in PyRIT. A separate `refusal_scorer/refusal_metrics.jsonl`
registry evaluates refusal scorers against its own human-labeled dataset, using the same
`ObjectiveScorerMetrics` shape. It isn't merged into the leaderboard above because it measures a
different task (refusal detection, not objective achievement) against a different ground truth
set, and mixing the two would make the F1 ranking misleading. A follow-up could add it as its own
leaderboard.